# Seq2Seq with Global Attention (Fast & Scaled)
This notebook introduces text cleaning and a Luong-style Global Attention layer to massively increase translation accuracy while maintaining fully vectorized cuDNN training speeds.


## 1. Import Libraries & GPU Setup


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import collections
import string
import re
import tensorflow.keras.preprocessing.sequence as ps

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled.")
    except RuntimeError as e:
        print(e)

tf.random.set_seed(42)
np.random.seed(42)


I0000 00:00:1783423806.195480   27722 cuda_dnn.cc:8703] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
I0000 00:00:1783423806.205278   27722 cuda_blas.cc:1413] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783423806.250795   27722 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783423806.250829   27722 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783423806.250831   27722 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783423806.250832   27722 computation_placer.cc:177] computation placer already registered. Please check linka

GPU Memory Growth Enabled.


## 2. Load & Clean Dataset
We strip all punctuation and lowercase the text to standardize our vocabularies and eliminate duplicate UNK-causing variants.


In [2]:
data = pd.read_csv('Dataset_English_Hindi.csv')
data.columns = ['English', 'Hindi']
data = data.dropna()

def clean_text(text):
    text = str(text).lower()
    # Remove punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning text...")
english_sentences = data['English'].apply(clean_text).tolist()
hindi_sentences = data['Hindi'].apply(clean_text).tolist()

print(f"Total cleaned sentences: {len(english_sentences)}")


Cleaning text...
Total cleaned sentences: 130162


## 3. Build Scaled Vocabularies
With clean text, we can safely scale the vocab size to 20,000 without hitting VRAM bottlenecks.


In [3]:
VOCAB_SIZE = 20000

def build_vocab(sentences, max_vocab_size):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    next_index = 4
    
    word_freq = collections.Counter()
    for sentence in sentences:
        word_freq.update(sentence.split())
        
    common_words = word_freq.most_common(max_vocab_size - len(vocab))
    
    for word, _ in common_words:
        vocab[word] = next_index
        next_index += 1
        
    return vocab

print("Building 20k vocabularies...")
eng_vocab = build_vocab(english_sentences, max_vocab_size=VOCAB_SIZE)
hind_vocab = build_vocab(hindi_sentences, max_vocab_size=VOCAB_SIZE)

with open('eng_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(eng_vocab, f, ensure_ascii=False)
with open('hind_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(hind_vocab, f, ensure_ascii=False)



Building 20k vocabularies...


## 4. Sequence Padding
Length scaled to 50 tokens.


In [4]:
MAX_LENGTH = 50

def sentence_to_number(sentence, vocab):
    words = sentence.split()
    unk_id = vocab['<UNK>']
    numbers = [vocab.get(word, unk_id) for word in words]
    return [vocab['<SOS>']] + numbers + [vocab['<EOS>']]

def prepare_data(eng_sentences, hind_sentences, eng_vocab, hind_vocab, max_length):
    src_number = [sentence_to_number(sent, eng_vocab) for sent in eng_sentences]
    tgt_number = [sentence_to_number(sent, hind_vocab) for sent in hind_sentences]

    src_padded = ps.pad_sequences(src_number, padding='post', maxlen=max_length, truncating='post', value=eng_vocab['<PAD>'])
    tgt_padded = ps.pad_sequences(tgt_number, padding='post', maxlen=max_length, truncating='post', value=hind_vocab['<PAD>'])
    return src_padded, tgt_padded

src_data, tgt_data = prepare_data(english_sentences, hindi_sentences, eng_vocab, hind_vocab, MAX_LENGTH)



## 5. Batching


In [5]:
BATCH_SIZE = 32 
BUFFER_SIZE = 10000 

dataset = tf.data.Dataset.from_tensor_slices((src_data, tgt_data))
dataset = dataset.shuffle(BUFFER_SIZE, seed=42).batch(BATCH_SIZE, drop_remainder=True)



I0000 00:00:1783423875.901621   27722 gpu_device.cc:2018] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0


## 6. Architecture with Vectorized Luong Attention


### 6.1 The Encoder
Now returns the full sequence of hidden states, which Attention needs to scan.


In [6]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_sequences=True, return_state=True)

    def call(self, x):
        embedded = self.embedding(x)
        enc_output, hidden, cell = self.lstm(embedded)
        return enc_output, hidden, cell



### 6.2 The Decoder with Global Attention
Processes the target sequence globally, then applies Attention across the entire Encoder output at once.


In [7]:
class DecoderWithAttention(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(DecoderWithAttention, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_sequences=True, return_state=True)
        # Keras built-in Luong-style dot-product Attention
        self.attention = tf.keras.layers.Attention()
        self.concat = tf.keras.layers.Concatenate(axis=-1)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, x, hidden, cell, enc_output):
        embedded = self.embedding(x)
        dec_output, h, c = self.lstm(embedded, initial_state=[hidden, cell])
        
        # Query = dec_output, Value = enc_output.
        # This calculates context globally over the entire sequence!
        context_vector = self.attention([dec_output, enc_output])
        
        combined = self.concat([dec_output, context_vector])
        output = self.dense(combined)
        return output, h, c



### 6.3 The Vectorized Seq2Seq Wrapper


In [8]:
class Seq2Seq(tf.keras.Model):
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size

    def call(self, inputs, training=False):
        src, tgt = inputs[0], inputs[1]
        enc_output, hidden, cell = self.encoder(src)
        
        if training:
            # Fully Vectorized Attention via Luong approach!
            decoder_input = tgt[:, :-1]
            output, _, _ = self.decoder(decoder_input, hidden, cell, enc_output)
            return output
            
        else:
            # Inference Autoregressive Loop
            batch_size = tf.shape(src)[0]
            tgt_len = tf.shape(tgt)[1] 
            
            i0 = tf.constant(1)
            input_token0 = tgt[:, 0:1] 
            all_outputs0 = tf.TensorArray(tf.float32, size=tgt_len - 1)
    
            def cond(i, input_token, h, c, all_outputs):
                return i < tgt_len
                
            def body(i, input_token, h, c, all_outputs):
                output, h, c = self.decoder(input_token, h, c, enc_output)
                all_outputs = all_outputs.write(i - 1, output)
                next_input = tf.argmax(output, axis=-1, output_type=tf.int32)
                return i + 1, next_input, h, c, all_outputs
    
            _, _, _, _, final_outputs = tf.while_loop(
                cond, body,
                loop_vars=[i0, input_token0, hidden, cell, all_outputs0]
            )
    
            outputs_stacked = final_outputs.stack()
            outputs_stacked = tf.squeeze(outputs_stacked, axis=2) 
            return tf.transpose(outputs_stacked, perm=[1, 0, 2]) 



## 7. Initialize Model


In [9]:
embed_size = 256
hidden_size = 512

encoder = Encoder(VOCAB_SIZE, embed_size, hidden_size)
decoder = DecoderWithAttention(VOCAB_SIZE, embed_size, hidden_size)
model = Seq2Seq(encoder, decoder, tgt_vocab_size=VOCAB_SIZE)

dummy_src = tf.zeros((BATCH_SIZE, MAX_LENGTH), dtype=tf.int32)
dummy_tgt = tf.zeros((BATCH_SIZE, MAX_LENGTH), dtype=tf.int32)
_ = model([dummy_src, dummy_tgt], training=True)
_ = model([dummy_src, dummy_tgt], training=False)



I0000 00:00:1783423903.751082   39215 cuda_dnn.cc:529] Loaded cuDNN version 92302


## 8. Training Logic


In [10]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

@tf.function
def train_step(src, tgt):
    with tf.GradientTape() as tape:
        predictions = model([src, tgt], training=True)
        target_labels = tgt[:, 1:] 

        mask = tf.cast(target_labels != hind_vocab["<PAD>"], tf.float32)
        loss = loss_fn(target_labels, predictions)
        loss = loss * mask

        total_loss = tf.reduce_sum(loss)
        total_token = tf.reduce_sum(mask)
        mean_loss = total_loss / total_token

    gradients = tape.gradient(mean_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return mean_loss



## 9. Execute Training Loop


In [11]:
EPOCHS = 30
PATIENCE = 3
best_loss = float('inf')
patience_counter = 0

print('Starting Scaled Attention Training...')

for epoch in range(EPOCHS):
    total_loss = 0.0
    steps = 0
    for batch_src, batch_tgt in dataset:
        loss = train_step(batch_src, batch_tgt)
        total_loss += float(loss.numpy())
        steps += 1
        
        if steps % 500 == 0:
            print(f"Epoch {epoch+1}, Step {steps}, Loss: {loss.numpy():.4f}")
            
    avg_loss = total_loss / steps
    print(f'\nEpoch: {epoch+1} | Average Loss: {avg_loss:.4f}')
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        model.save_weights('seq2seq_attention.weights.h5')
        print(f"[*] Loss improved to {best_loss:.4f}. Model weights saved.\n")
    else:
        patience_counter += 1
        print(f"[!] No improvement in loss. Patience: {patience_counter}/{PATIENCE}\n")
        if patience_counter >= PATIENCE:
            print("=> Early stopping triggered! Training stopped.")
            break



Starting Scaled Attention Training...
Epoch 1, Step 500, Loss: 6.3763
Epoch 1, Step 1000, Loss: 5.7203
Epoch 1, Step 1500, Loss: 5.1356
Epoch 1, Step 2000, Loss: 5.0170
Epoch 1, Step 2500, Loss: 5.1097
Epoch 1, Step 3000, Loss: 4.5190
Epoch 1, Step 3500, Loss: 4.3313
Epoch 1, Step 4000, Loss: 3.9018

Epoch: 1 | Average Loss: 5.2061
[*] Loss improved to 5.2061. Model weights saved.

Epoch 2, Step 500, Loss: 3.4885
Epoch 2, Step 1000, Loss: 4.0788
Epoch 2, Step 1500, Loss: 3.9569
Epoch 2, Step 2000, Loss: 3.7314
Epoch 2, Step 2500, Loss: 3.4282
Epoch 2, Step 3000, Loss: 3.3177
Epoch 2, Step 3500, Loss: 3.3385
Epoch 2, Step 4000, Loss: 3.1085

Epoch: 2 | Average Loss: 3.7053
[*] Loss improved to 3.7053. Model weights saved.

Epoch 3, Step 500, Loss: 3.2698
Epoch 3, Step 1000, Loss: 3.1222
Epoch 3, Step 1500, Loss: 3.2621
Epoch 3, Step 2000, Loss: 2.9049
Epoch 3, Step 2500, Loss: 2.3666
Epoch 3, Step 3000, Loss: 3.2088
Epoch 3, Step 3500, Loss: 2.7247
Epoch 3, Step 4000, Loss: 2.6967

Epoc